# Use models through Models-as-a-Service

This AI namespace was ordered through the self-service catalog. The MaaS API key is already in the workbench, as environment variables from the secret `maas-access`. You don't have to set up any of it yourself.

In [ ]:
import os, requests

base_url = os.environ["MAAS_BASE_URL"]
headers = {"Authorization": f"Bearer {os.environ['MAAS_API_KEY']}"}

# Which models may this namespace use?
models = requests.get(f"{base_url}/models", headers=headers, timeout=30).json()
[m["id"] for m in models.get("data", [])]

## Ask a model a question

The same OpenAI compatible API for the self-hosted Granite and for external models like Claude Haiku. Only the model name changes.

In [ ]:
def ask(model, text, max_tokens=200):
    r = requests.post(f"{base_url}/chat/completions", headers=headers, timeout=120, json={
        "model": model,
        "messages": [{"role": "user", "content": text}],
        "max_tokens": max_tokens,
    })
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

print(ask(os.environ.get("MAAS_MODEL", "granite"), "Explain in two sentences what Models-as-a-Service is."))

## Mini RAG: answers from your own text

No vector database, just to show the principle: we hand the model an excerpt from an internal document and let it answer only from that.

In [ ]:
document = """
Platform team operations handbook (excerpt):
- You order namespaces in the Ansible Automation Portal or through an agent over MCP.
- Databases cost money and need an approval in AAP.
- Edge clusters pull their configuration from Git themselves (pull model).
"""

print(ask(os.environ.get("MAAS_MODEL", "granite"),
          f"Answer only from this text: How do I get a database?\n\n{document}"))